# Aftershoot AI Image Classification Challenge

## Importing essential packages and libraries for Model Development

In [2]:
import pandas as pd
import numpy as np
from PIL import Image
import os
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

**Retrieval of the location of the Training and Validation datasets**

In [4]:
# Configuration
TRAIN_DIR = 'dataset/Train'
VAL_DIR = 'dataset/Validation'
IMAGE_SIZE = (256, 256)

## Defining Functions

### Preprocessing and Loading the images.

In [5]:
def load_and_preprocess_image(img_path):
    """Load TIFF image and extract features"""
    try:
        img = Image.open(img_path)
        img = img.resize(IMAGE_SIZE)
        img_array = np.array(img)
        
        # Extract statistical features from image
        features = []
        
        # Handle grayscale or RGB
        if len(img_array.shape) == 2:
            img_array = np.stack([img_array] * 3, axis=-1)
        
        # Extract features for each channel
        for channel in range(3):
            channel_data = img_array[:, :, channel].flatten()
            features.extend([
                np.mean(channel_data),
                np.std(channel_data),
                np.median(channel_data),
                np.percentile(channel_data, 25),
                np.percentile(channel_data, 75),
                np.min(channel_data),
                np.max(channel_data)
            ])
        
        # Overall image features
        gray = np.mean(img_array, axis=2)
        features.extend([
            np.mean(gray),
            np.std(gray),
            np.median(gray)
        ])
        
        return features
    except Exception as e:
        print(f"Error loading image {img_path}: {e}")
        return None



### Preparing Features

In [6]:
def prepare_features(df, img_dir, is_training=True):
    """Prepare feature matrix from CSV and images"""
    X_list = []
    valid_indices = []
    
    for idx, row in df.iterrows():
        img_id = row['id_global']
        img_path = os.path.join(img_dir, f"{img_id}.tif")
        
        if not os.path.exists(img_path):
            img_path = os.path.join(img_dir, f"{img_id}.tiff")
        
        if os.path.exists(img_path):
            img_features = load_and_preprocess_image(img_path)
            
            if img_features is not None:
                # Combine image features with CSV features
                csv_features = []
                
                # As Shot WB
                if 'currTemp' in row:
                    csv_features.append(row['currTemp'])
                if 'currTint' in row:
                    csv_features.append(row['currTint'])
                
                # EXIF data
                exif_cols = ['aperture', 'flashFired', 'focalLength', 
                            'isoSpeedRating', 'shutterSpeed']
                for col in exif_cols:
                    if col in row:
                        csv_features.append(row[col])
                
                # Camera info - encode as needed
                if 'camera_model' in row:
                    # For simplicity, hash the model name
                    csv_features.append(hash(str(row['camera_model'])) % 10000)
                if 'camera_group' in row:
                    csv_features.append(hash(str(row['camera_group'])) % 1000)
                
                # Extra properties
                if 'intensity' in row:
                    csv_features.append(row['intensity'])
                if 'ev' in row:
                    csv_features.append(row['ev'])
                
                # Combine all features
                all_features = img_features + csv_features
                X_list.append(all_features)
                valid_indices.append(idx)
        
        if (idx + 1) % 100 == 0:
            print(f"Processed {idx + 1}/{len(df)} images")
    
    X = np.array(X_list)
    valid_df = df.iloc[valid_indices].reset_index(drop=True)
    
    return X, valid_df



### Model Training from the Train Dataset

In [7]:
def train_model():
    """Train the model on training data"""
    print("Loading training data...")
    train_csv = pd.read_csv(os.path.join(TRAIN_DIR, 'sliders.csv'))
    train_img_dir = os.path.join(TRAIN_DIR, 'images')
    
    print("Extracting features from training images...")
    X_train, train_df = prepare_features(train_csv, train_img_dir, is_training=True)
    
    # Target variables
    y_temp = train_df['Temperature'].values
    y_tint = train_df['Tint'].values
    
    print(f"Training data shape: {X_train.shape}")
    print(f"Target Temperature range: [{y_temp.min()}, {y_temp.max()}]")
    print(f"Target Tint range: [{y_tint.min()}, {y_tint.max()}]")
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    # Train models
    print("\nTraining Temperature model...")
    temp_model = GradientBoostingRegressor(
        n_estimators=200,
        max_depth=8,
        learning_rate=0.1,
        random_state=42,
        subsample=0.8
    )
    temp_model.fit(X_train_scaled, y_temp)
    
    print("Training Tint model...")
    tint_model = GradientBoostingRegressor(
        n_estimators=200,
        max_depth=8,
        learning_rate=0.1,
        random_state=42,
        subsample=0.8
    )
    tint_model.fit(X_train_scaled, y_tint)
    
    # Cross-validation scores
    print("\nEvaluating models...")
    temp_cv = -cross_val_score(temp_model, X_train_scaled, y_temp, 
                               cv=3, scoring='neg_mean_absolute_error')
    tint_cv = -cross_val_score(tint_model, X_train_scaled, y_tint, 
                               cv=3, scoring='neg_mean_absolute_error')
    
    print(f"Temperature MAE (CV): {temp_cv.mean():.2f} (+/- {temp_cv.std():.2f})")
    print(f"Tint MAE (CV): {tint_cv.mean():.2f} (+/- {tint_cv.std():.2f})")
    
    return temp_model, tint_model, scaler



### Predictions Generation

In [8]:
def generate_predictions(temp_model, tint_model, scaler):
    """Generate predictions for validation set"""
    print("\nLoading validation data...")
    val_csv = pd.read_csv(os.path.join(VAL_DIR, 'sliders_input.csv'))
    val_img_dir = os.path.join(VAL_DIR, 'images')
    
    print("Extracting features from validation images...")
    X_val, val_df = prepare_features(val_csv, val_img_dir, is_training=False)
    
    print(f"Validation data shape: {X_val.shape}")
    
    # Scale features
    X_val_scaled = scaler.transform(X_val)
    
    # Make predictions
    print("Making predictions...")
    temp_pred = temp_model.predict(X_val_scaled)
    tint_pred = tint_model.predict(X_val_scaled)
    
    # Round to nearest integer
    temp_pred = np.round(temp_pred).astype(int)
    tint_pred = np.round(tint_pred).astype(int)
    
    # Create submission dataframe
    submission = pd.DataFrame({
        'id_global': val_df['id_global'],
        'Temperature': temp_pred,
        'Tint': tint_pred
    })
    
    # Ensure we have exactly 493 predictions
    if len(submission) < 493:
        print(f"Warning: Only {len(submission)} predictions generated instead of 493")
    
    return submission



### Main Function

In [9]:
def main():
    """Main execution function"""
    print("="*60)
    print("Aftershoot ML Model - Temperature & Tint Prediction")
    print("="*60)
    
    # Train models
    temp_model, tint_model, scaler = train_model()
    
    # Generate predictions
    submission = generate_predictions(temp_model, tint_model, scaler)
    
    # Save predictions
    output_file = 'predictions.csv'
    submission.to_csv(output_file, index=False)
    
    print(f"\nPredictions saved to {output_file}")
    print(f"Submission shape: {submission.shape}")
    print("\nFirst few predictions:")
    print(submission.head())
    print("\nSummary statistics:")
    print(submission.describe())
    
    print("\n" + "="*60)
    print("Processing complete!")
    print("="*60)

if __name__ == "__main__":
    main()

Aftershoot ML Model - Temperature & Tint Prediction
Loading training data...
Extracting features from training images...
Processed 100/2538 images
Processed 200/2538 images
Processed 300/2538 images
Processed 400/2538 images
Processed 500/2538 images
Processed 600/2538 images
Processed 700/2538 images
Processed 800/2538 images
Processed 900/2538 images
Processed 1000/2538 images
Processed 1100/2538 images
Processed 1200/2538 images
Processed 1300/2538 images
Processed 1400/2538 images
Processed 1500/2538 images
Processed 1600/2538 images
Processed 1700/2538 images
Processed 1800/2538 images
Processed 1900/2538 images
Processed 2000/2538 images
Processed 2100/2538 images
Processed 2200/2538 images
Processed 2300/2538 images
Processed 2400/2538 images
Processed 2500/2538 images
Training data shape: (2538, 31)
Target Temperature range: [2000, 49200]
Target Tint range: [-90, 40]

Training Temperature model...
Training Tint model...

Evaluating models...
Temperature MAE (CV): 402.05 (+/- 27